In [ ]:
# init
import importlib, sys

import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

import superconductivity.api as sc

from superconductivity.api import G0_muS
from superconductivity.api import NDArray64

from IPython import get_ipython

from tqdm import tqdm

Textwidth: float = 4.25279  # in
Textheight: float = 6.85173  # in

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic("config", "InlineBackend.rc = {'figure.dpi': 300}")

# breaking

In [ ]:
# # seetings
# data = np.load("breaking/eva.npz")
# V_mV = data["Vbias_mV"]
# Iexp_nA = data["Iup_nA"]
# x_arbu = data["x_arbu"]
# settings = {
#     "tau_1": (0.01, 0.00, 1.00, False),
#     "tau_2": (0.01, 0.00, 1.00, False),
#     "tau_3": (0.01, 0.00, 1.00, False),
#     "tau_4": (0.01, 0.00, 1.00, False),
#     "tau_5": (0.01, 0.00, 1.00, False),
#     "tau_6": (0.01, 0.00, 1.00, False),
#     "tau_7": (0.01, 0.00, 1.00, False),
#     "tau_8": (0.00, 0.00, 1.00, False),
#     "tau_9": (0.00, 0.00, 1.00, False),
#     "tau_A": (0.00, 0.00, 1.00, True),
#     "tau_B": (0.00, 0.00, 1.00, True),
#     "tau_C": (0.00, 0.00, 1.00, True),
#     "T_K": (0.00, 0.00, 1.21, True),
#     "Delta_meV": (0.1885, 0.187, 0.191, False),
#     "gamma_meV": (1e-6, 1e-6, 1e-2, True),
#     "sigmaV_mV": (0.026, 0.01, 0.04, False),
# }
# restarts = 10
# Vnan_mV = 0.04
# tau_sum_bounds = (0.0, 5.0)

In [ ]:
# # fit all data (steepest gradient)
# import numpy as np
# from mar_fit import fit_mar

# # load data

# weights = np.where(1 / V_mV, V_mV != 0, 0.0)
# mask = np.abs(V_mV) <= Vnan_mV
# I_nA = np.copy(Iexp_nA)
# I_nA[:, mask] = np.nan
# results = fit_mar(
#     I_nA[:3, :],
#     "grid.pkl",
#     settings=settings,
#     tau_sum_bounds=tau_sum_bounds,
#     restarts=restarts,
#     progress=True,
# )
# Ifit_nA = np.full_like(I_nA)
# taus = np.full((I_nA.shape[0], 12), np.nan)
# T_K = np.full((I_nA.shape[0]), np.nan)
# Delta_meV = np.full((I_nA.shape[0]), np.nan)
# gamma_meV = np.full((I_nA.shape[0]), np.nan)
# sigmaV_mV = np.full((I_nA.shape[0]), np.nan)

# for i, result in enumerate(results):
#     Ifit_nA[i, :] = result.Ifit_nA
#     T_K[i] = result.T_K
#     Delta_meV[i, :] = result.Delta_meV
#     gamma_meV[i, :] = result.gamma_meV
#     sigmaV_mV[i, :] = result.sigmaV_mV

# GN_G0 = np.nansum(taus, axis=-1)
# np.savez(
#     "breaking/fit_data.npz",
#     V_mV=V_mV,
#     x_arbu=x_arbu,
#     Iexp_nA=I_nA,
#     Ifit_nA=Ifit_nA,
#     taus=taus,
#     GN_G0=GN_G0,
#     T_K=T_K,
#     Delta_meV=Delta_meV,
#     gamma_meV=gamma_meV,
#     sigmaV_mV=sigmaV_mV,
#     results=results,
#     mask=mask,
#     weights=weights,
#     settings=settings,
#     tau_sum_bounds=tau_sum_bounds,
# )

# $550 G_0$

In [ ]:
data = np.load(f"550G0/magnetic-field/eva.npz")
Vbias_mV = data["Vbias_mV"]
Ibias_nA = data["Ibias_nA"]
Tbath_K = data["Tbath_K"]
dGexp_G0 = data["dGexp_G0"]
dRexp_R0 = data["dRexp_R0"]
Iexp_nA = data["Iexp_nA"]
Vexp_mV = data["Vexp_mV"]
uH_mT = data["uH_mT"]

uHc_mT = 9.95
Tc_K = 1.21
Ic_nA = 1.075e6
RN_Ohm = 23.5

uH_mT += 0.55
uH = uH_mT / uHc_mT
Vbias = Vbias_mV * 1e6 / RN_Ohm / Ic_nA
Vexp = Vexp_mV * 1e6 / RN_Ohm / Ic_nA
Ibias = Ibias_nA / Ic_nA
Iexp = Iexp_nA / Ic_nA
dGexp = dGexp_G0 * sc.G0_muS * 1e-6 * RN_Ohm
dRexp = dRexp_R0 / sc.G0_muS * 1e6 / RN_Ohm

np.savez(
    f"550G0/magnetic-field/cal.npz",
    Vbias=Vbias,
    Ibias=Ibias,
    uH=uH,
    Iexp=Iexp,
    Vexp=Vexp,
    dGexp=dGexp,
    dRexp=dRexp,
)

# $0.05 G_0$

In [ ]:
# # temperature, magnetic field
data = np.load(f"0.05G0/temperature/eva.npz")
Vbias_mV = data["Vbias_mV"]
Ibias_nA = data["Ibias_nA"]
Tbath_K = data["Tbath_K"]
dGexp_G0 = data["dGexp_G0"]
dRexp_R0 = data["dRexp_R0"]
Iexp_nA = data["Iexp_nA"]

Tc_K = 1.21
Delta_meV = 0.1885
GN_G0 = 0.13367895134953922

Vbias = Vbias_mV / Delta_meV
Ibias = Ibias_nA / (Delta_meV * GN_G0 * sc.G0_muS)
Tbath = Tbath_K / Tc_K
Iexp = Iexp_nA / (Delta_meV * GN_G0 * sc.G0_muS)
dGexp = dGexp_G0 / GN_G0
dRexp = dRexp_R0 * GN_G0

np.savez(
    f"0.05G0/temperature/cal.npz",
    Vbias=Vbias,
    Ibias=Ibias,
    Tbath=Tbath,
    Iexp=Iexp,
    dGexp=dGexp,
    dRexp=dRexp,
)

data = np.load(f"0.05G0/magnetic-field/eva.npz")
Vbias_mV = data["Vbias_mV"]
Ibias_nA = data["Ibias_nA"]
Tbath_K = data["Tbath_K"]
dGexp_G0 = data["dGexp_G0"]
dRexp_R0 = data["dRexp_R0"]
Iexp_nA = data["Iexp_nA"]
uH_mT = data["uH_mT"]

uHc_mT = 28
Tc_K = 1.21
Delta_meV = 0.1885
GN_G0 = 0.04837291812646421

Vbias = Vbias_mV / Delta_meV
Ibias = Ibias_nA / (Delta_meV * GN_G0 * sc.G0_muS)
uH = uH_mT / uHc_mT
Iexp = Iexp_nA / (Delta_meV * GN_G0 * sc.G0_muS)
dGexp = dGexp_G0 / GN_G0
dRexp = dRexp_R0 * GN_G0

np.savez(
    f"0.05G0/magnetic-field/cal.npz",
    Vbias=Vbias,
    Ibias=Ibias,
    uH=uH,
    Iexp=Iexp,
    dGexp=dGexp,
    dRexp=dRexp,
)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib qt
# plt.imshow(
#     dGexp,
#     aspect="auto",
#     interpolation="None",
#     origin="lower",
#     extent=(
#         np.min(Vbias_mV),
#         np.max(Vbias_mV),
#         np.min(uH_mT),
#         np.max(uH_mT),
#     ),
#     clim=(0,2),
# )
plt.imshow(
    dGexp,
    aspect="auto",
    interpolation="None",
    origin="lower",
    extent=(
        np.min(Vbias),
        np.max(Vbias),
        np.min(uH),
        np.max(uH),
    ),
    clim=(0, 2),
)
# plt.imshow(
#     dRexp,
#     aspect="auto",
#     interpolation="None",
#     origin="lower",
#     extent=(
#         np.min(Ibias_nA),
#         np.max(Ibias_nA),
#         np.min(uH_mT),
#         np.max(uH_mT),
#     ),
#     clim=(0, 2),
# )
# plt.imshow(
#     dRexp,
#     aspect="auto",
#     interpolation="None",
#     origin="lower",
#     extent=(
#         np.min(Ibias),
#         np.max(Ibias),
#         np.min(uH),
#         np.max(uH),
#     ),
#     clim=(0, 2),
# )
plt.show()